# 17 · Partitions, shuffles, caching, lazy evaluation (Spark)

- A **partition** = one chunk of a DataFrame. One **task** processes one partition on one CPU core.
- **Narrow** transformation (`filter`, `select`, `withColumn`): each output partition needs only one input
  partition. No data moves.
- **Wide** transformation (`groupBy`, `join`, `distinct`, `orderBy`, `repartition`): rows must move between
  partitions = a **shuffle** (`Exchange` in the plan). Shuffles split a job into **stages**.
- **AQE** (Adaptive Query Execution) changes the plan while running: merges small shuffle partitions,
  switches to broadcast joins, splits skewed partitions.

This practice Spark uses `local[*]` (all your cores) and `spark.sql.shuffle.partitions = 4`.

**How to run:** Start a cell with `%%sql` to write Spark SQL. In a normal Python cell use the DataFrame API: the tables are ready as variables (`employees`, `orders`, ...). Put your answer in `res` and call `res.show()`.

Tables: see [tables_guide.md](../../jarvis/tables_guide.md). Run the setup cell first (Shift+Enter).

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "settings.py").exists())   # the repo folder
sys.path.insert(0, str(ROOT / "spark"))
from spark_setup import get_spark, load_tables
from pyspark.sql import functions as F, Window, types as T

spark = get_spark()
t = load_tables(spark)          # also makes SQL views with the same names
employees, departments = t["employees"], t["departments"]
customers, orders = t["customers"], t["orders"]
products, order_items = t["products"], t["order_items"]
user_logins = t["user_logins"]
%load_ext spark_setup

## Q1. How many partitions?  ·  🟢 easy

Print the number of partitions of: `orders` (read from a file), `orders.repartition(8)`, and `spark.range(0, 1000, numPartitions=6)`.

In [ ]:
# your answer here (DataFrame API)

## Q2. repartition vs coalesce  ·  🟡 medium

Start from `spark.range(1000, numPartitions=8)`. Make 2 partitions with `coalesce(2)` and with `repartition(2)`. Print both plans: which one has an `Exchange` (shuffle)?

In [ ]:
# your answer here (DataFrame API)

## Q3. Rows per partition  ·  🟡 medium

Repartition `orders` by `customer_id` into 4 partitions and count rows per partition with `F.spark_partition_id()`.

In [ ]:
# DataFrame API: put your answer in res
res = ...
res.show()

## Q4. Narrow vs wide: find the shuffle  ·  🟢 easy

Print the plan of: orders → filter completed → withColumn year → groupBy year → count. Which step makes the `Exchange`?

In [ ]:
# your answer here (DataFrame API)

## Q5. Lazy evaluation  ·  🟢 easy

Build a DataFrame with a filter and a new column, and time it. Then call `count()` and time that. Which one actually did the work?

In [ ]:
# your answer here (DataFrame API)

## Q6. cache and persist  ·  🟡 medium

Make an expensive DataFrame (`spark.range(3_000_000)` with a groupBy), `cache()` it, run `count()` twice and time both. Check `is_cached`, then `unpersist()`.

In [ ]:
# your answer here (DataFrame API)

## Q7. Shuffle partitions and AQE  ·  🟡 medium

Print `spark.sql.shuffle.partitions` and `spark.sql.adaptive.enabled`. Then run a groupBy on `orders` and print the output partition count with AQE **on** and **off**.

In [ ]:
# your answer here (DataFrame API)

## Q8. Cut a long lineage with checkpoint  ·  🔴 hard

Run a loop that adds a column 30 times. Print the length of the plan text before and after `localCheckpoint()`.

In [ ]:
# your answer here (DataFrame API)

## Q9. Spark SQL hints for partitions  ·  🟢 easy

In Spark SQL, use the `REPARTITION(3)` and `COALESCE(1)` hints on `orders` and check the partition count.

In [ ]:
%%sql
-- your answer here

In [ ]:
# your answer here (DataFrame API)